[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladimiracunadev-create/python-data-science-bootcamp/blob/master/classes/11-evaluacion-y-pipelines/notebook.ipynb)

# Clase 11 — Evaluacion Robusta y Pipelines de ML

---

## Objetivos de la clase

Al terminar esta clase seras capaz de:

- **Entender** por que un solo split de train/test no es suficiente para evaluar un modelo
- **Aplicar** Cross-Validation con `StratifiedKFold` para obtener estimaciones confiables
- **Construir** Pipelines que encadenen preprocesamiento y modelo de forma segura
- **Optimizar** hiperparametros automaticamente con `GridSearchCV`
- **Diagnosticar** overfitting y underfitting usando curvas de aprendizaje


## El problema con un solo train/test split

```
PROBLEMA CON UN SOLO SPLIT
===========================

Dataset de 200 filas. Si hacemos el split una sola vez:

  Split A (seed=1):  Test accuracy = 0.81  <- suerte con este test set?
  Split B (seed=2):  Test accuracy = 0.74  <- mala suerte con este?
  Split C (seed=3):  Test accuracy = 0.87  <- cual es el verdadero?

  El resultado depende de CUALES filas terminaron en el test set.
  Si por azar caen filas "faciles" -> score inflado.
  Si por azar caen filas "dificiles" -> score deflado.

  Pregunta: ¿cual de estos 3 scores reportamos al cliente?
  Respuesta: NINGUNO por si solo es confiable.

SOLUCION: Cross-Validation
  Evaluar el modelo K veces, cada vez con un test set diferente.
  Reportar la MEDIA y la DESVIACION ESTANDAR de los K scores.
```


In [ ]:
# ============================================================
# BLOQUE: Importacion de librerias para evaluacion y pipelines
# ============================================================

# --- Manipulacion y calculo numerico ---
import pandas as pd   # DataFrames y manipulacion de tablas
import numpy as np    # arrays y operaciones matematicas

# --- Visualizacion ---
import matplotlib.pyplot as plt  # graficos de linea, barra, etc.

# --- Modelos ---
from sklearn.tree import DecisionTreeClassifier      # arbol de decision
from sklearn.linear_model import LogisticRegression  # regresion logistica

# --- Pipeline: encadenar pasos de ML de forma segura ---
from sklearn.pipeline import Pipeline  # objeto que conecta transformadores + modelo

# --- ColumnTransformer: aplicar distintas transformaciones por columna ---
from sklearn.compose import ColumnTransformer  # transformaciones diferenciadas por columna

# --- Herramientas de evaluacion y busqueda ---
from sklearn.model_selection import train_test_split  # division train/test
from sklearn.model_selection import cross_val_score   # evaluacion con K folds
from sklearn.model_selection import StratifiedKFold   # K-Fold que preserva balance de clases
from sklearn.model_selection import GridSearchCV      # busqueda exhaustiva de hiperparametros
from sklearn.model_selection import learning_curve    # curva train vs val por tamano

# --- Preprocesamiento ---
from sklearn.preprocessing import StandardScaler  # escalado a media=0, std=1

# --- Metricas ---
from sklearn.metrics import classification_report  # reporte completo
from sklearn.metrics import f1_score               # F1 individual

print("Librerias cargadas correctamente")  # confirmar importaciones exitosas


In [ ]:
# ============================================================
# BLOQUE: Carga de datos y preparacion de X e y
# ============================================================

# Paso 1: cargar el mismo dataset de retencion de la clase anterior
df = pd.read_csv("../../datasets/retencion_clientes.csv")  # leer CSV
print(f"Dataset cargado: {df.shape}")

# Paso 2: seleccionar solo columnas numericas como features
# (mismo procedimiento que en la clase 10 para mantener consistencia)
cols_num = df.select_dtypes(include=["number"]).columns.tolist()  # solo numericas
features = [c for c in cols_num if c != "churned"]  # excluir el target

# Paso 3: crear X e y
X = df[features]       # matrix de features
y = df["churned"]      # vector target binario (0/1)

print(f"Features: {features}")
print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

# Paso 4: IMPORTANTE - separar un holdout set final ANTES de cualquier CV
# El holdout es el 20% que NUNCA tocamos durante el desarrollo
# Solo lo usamos al final para la evaluacion definitiva
X_cv, X_holdout, y_cv, y_holdout = train_test_split(
    X, y,
    test_size=0.20,   # 20% reservado como holdout final
    random_state=42,  # reproducibilidad
    stratify=y        # mantener balance de clases
)

print(f"\nDatos para Cross-Validation: {X_cv.shape[0]} filas")
print(f"Holdout final (intocable):   {X_holdout.shape[0]} filas")


## K-Fold Cross Validation

```
K-FOLD CROSS VALIDATION (K=5)
==============================

Dataset (200 filas divididas en 5 grupos iguales de 40 filas)

+--------+--------+--------+--------+--------+
| Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
+--------+--------+--------+--------+--------+

Iteracion 1: [ TEST  ][ TRAIN ][ TRAIN ][ TRAIN ][ TRAIN ]  score_1 = 0.82
Iteracion 2: [ TRAIN ][ TEST  ][ TRAIN ][ TRAIN ][ TRAIN ]  score_2 = 0.79
Iteracion 3: [ TRAIN ][ TRAIN ][ TEST  ][ TRAIN ][ TRAIN ]  score_3 = 0.84
Iteracion 4: [ TRAIN ][ TRAIN ][ TRAIN ][ TEST  ][ TRAIN ]  score_4 = 0.81
Iteracion 5: [ TRAIN ][ TRAIN ][ TRAIN ][ TRAIN ][ TEST  ]  score_5 = 0.80

Score final = mean(0.82, 0.79, 0.84, 0.81, 0.80) = 0.812 +/- 0.018

Ventajas:
  - TODOS los datos se usan como train Y como test
  - El score es mucho mas estable y confiable
  - La desviacion estandar nos dice si el modelo es consistente

StratifiedKFold: version que garantiza que la proporcion de clases
se mantiene igual en cada fold (importante cuando hay desbalance)
```


In [ ]:
# ============================================================
# BLOQUE: Cross-Validation con StratifiedKFold
# ============================================================

# Paso 1: crear el modelo base que vamos a evaluar
# Usamos DecisionTree con parametros moderados como punto de partida
modelo_base = DecisionTreeClassifier(
    max_depth=5,    # profundidad moderada para evitar overfitting obvio
    random_state=42 # reproducibilidad
)

# Paso 2: definir la estrategia de division
# StratifiedKFold preserva la proporcion de clases en cada fold
# n_splits=5 -> 5 iteraciones, cada una usa 20% como test
# shuffle=True -> mezclar los datos antes de dividir (evita patrones de orden)
kfold = StratifiedKFold(
    n_splits=5,     # K=5 folds (estandar en la industria)
    shuffle=True,   # mezclar antes de dividir para mayor aleatoriedad
    random_state=42 # reproducibilidad de la mezcla
)

# Paso 3: ejecutar la evaluacion cruzada
# cross_val_score entrena y evalua el modelo K veces automaticamente
# scoring='f1' usa F1 de la clase 1 como metrica principal
# NOTA: cross_val_score no hace fit en datos de test, previene data leakage
cv_scores = cross_val_score(
    modelo_base,  # objeto del modelo a evaluar
    X_cv,         # features del conjunto de CV (sin holdout)
    y_cv,         # target del conjunto de CV
    cv=kfold,     # estrategia de division K-Fold
    scoring="f1"  # metrica: F1 de la clase positiva (churners)
)

# Paso 4: mostrar los scores de cada fold y el resumen estadistico
print("Scores F1 por fold:")
for i, s in enumerate(cv_scores, start=1):
    barra = "=" * int(s * 30)  # barra proporcional al score
    print(f"  Fold {i}: {s:.4f}  |{barra}|")

print()
print(f"Media:              {cv_scores.mean():.4f}")
print(f"Desviacion estandar: {cv_scores.std():.4f}")
print(f"Intervalo confianza: [{cv_scores.mean()-cv_scores.std():.4f}, {cv_scores.mean()+cv_scores.std():.4f}]")
print()
print("INTERPRETACION: El modelo tiene un F1 esperado de")
print(f"  {cv_scores.mean():.4f} con una variabilidad de +/- {cv_scores.std():.4f}")


In [ ]:
# ============================================================
# BLOQUE: Visualizar scores de cada fold como grafico de barras
# ============================================================

# Paso 1: preparar los datos para el grafico
folds = [f"Fold {i}" for i in range(1, len(cv_scores) + 1)]  # etiquetas del eje X

# Colorear barras: azul si el fold supera la media, salmon si esta por debajo
colores = ["steelblue" if s >= cv_scores.mean() else "salmon" for s in cv_scores]

# Paso 2: crear la figura
fig, ax = plt.subplots(figsize=(8, 5))  # lienzo de 8x5 pulgadas

# Paso 3: dibujar las barras
barras = ax.bar(
    folds,          # posiciones en el eje X
    cv_scores,      # alturas de las barras (valores de F1)
    color=colores,  # color diferenciado por rendimiento
    edgecolor="white",  # borde blanco para separacion visual
    width=0.6       # ancho de cada barra
)

# Paso 4: linea horizontal de la media
ax.axhline(
    y=cv_scores.mean(),     # posicion vertical de la linea
    color="darkblue",       # color diferenciador
    linewidth=2.5,          # grosor de la linea
    linestyle="--",         # estilo punteado
    label=f"Media = {cv_scores.mean():.4f}"  # leyenda con el valor
)

# Paso 5: etiquetas con el valor encima de cada barra
for barra, score in zip(barras, cv_scores):
    ax.text(
        barra.get_x() + barra.get_width() / 2,  # centro horizontal de la barra
        score + 0.003,                           # ligeramente encima de la barra
        f"{score:.3f}",                          # valor con 3 decimales
        ha="center", va="bottom",                # alineacion centrada
        fontsize=10, fontweight="bold"           # texto bold y legible
    )

# Paso 6: configurar ejes y titulo
ax.set_xlabel("Fold de Cross-Validation", fontsize=12)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_title("Cross-Validation: F1 Score por Fold\n(Azul = sobre la media, Salmon = bajo la media)",
             fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.1)         # eje Y de 0 a 1.1 para dar espacio a las etiquetas
ax.legend(fontsize=11)       # mostrar leyenda de la linea de media
ax.grid(True, alpha=0.3, axis="y")  # cuadricula suave solo en eje Y

plt.tight_layout()  # ajustar margenes
plt.show()          # mostrar el grafico


## Diagnostico de Overfitting vs Underfitting

```
DIAGNOSTICO DE OVERFITTING
===========================

Train Score  |  Test Score  |  Brecha   |  Diagnostico
-------------|--------------|-----------|-------------------------------------------
    0.90     |    0.88      | pequena   |  OK  Generaliza bien
    0.95     |    0.85      | media     |  AVISO  Revisar complejidad del modelo
    0.99     |    0.65      | grande    |  ERROR  Overfitting grave (memorizo train)
    0.65     |    0.63      | muy peq.  |  ERROR  Underfitting (modelo muy simple)

CAUSAS Y SOLUCIONES:

  Overfitting:
    Causa:    modelo demasiado complejo para los datos disponibles
    Solucion: reducir max_depth, aumentar min_samples_leaf,
              conseguir mas datos, usar regularizacion

  Underfitting:
    Causa:    modelo demasiado simple para capturar los patrones
    Solucion: aumentar max_depth, agregar mas features,
              usar un modelo mas complejo
```


In [ ]:
# ============================================================
# BLOQUE: Diagnostico de overfitting variando max_depth
# ============================================================

# Paso 1: definir el rango de profundidades a explorar
# Probaremos desde arboles muy simples (depth=1) hasta muy complejos (depth=15)
profundidades = list(range(1, 16))  # [1, 2, 3, ..., 15]

# Listas para almacenar los scores de cada profundidad
train_scores = []  # scores en datos de entrenamiento
test_scores  = []  # scores en datos de prueba (CV)

# Paso 2: hacer un split simple para medir train vs test
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cv, y_cv, test_size=0.2, random_state=42, stratify=y_cv
)

# Paso 3: entrenar y evaluar para cada profundidad
for depth in profundidades:
    # Crear arbol con la profundidad actual
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_tr, y_tr)  # entrenar con datos de train

    # Score en TRAIN: mide cuanto memorizo el modelo
    train_f1 = f1_score(y_tr, dt.predict(X_tr), average="weighted")  # F1 en train
    # Score en TEST: mide cuanto generaliza el modelo
    test_f1  = f1_score(y_te, dt.predict(X_te), average="weighted")  # F1 en test

    train_scores.append(train_f1)  # guardar resultado de train
    test_scores.append(test_f1)    # guardar resultado de test

# Paso 4: graficar ambas curvas
fig, ax = plt.subplots(figsize=(10, 6))  # lienzo amplio

# Curva de train (azul): muestra cuanto aprende el modelo
ax.plot(profundidades, train_scores, "o-", color="steelblue",
        label="Train F1", linewidth=2, markersize=6)

# Curva de test (naranja): muestra cuanto generaliza el modelo
ax.plot(profundidades, test_scores, "s--", color="darkorange",
        label="Test F1", linewidth=2, markersize=6)

# Zona de overfitting: cuando train >> test
ax.fill_between(profundidades, train_scores, test_scores,
                alpha=0.1, color="red", label="Zona de over-fitting")

ax.set_xlabel("Profundidad del Arbol (max_depth)", fontsize=12)
ax.set_ylabel("F1 Score (weighted)", fontsize=12)
ax.set_title("Overfitting: Train vs Test Score por max_depth",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)  # cuadricula suave
ax.set_xticks(profundidades)  # mostrar cada valor de depth en el eje X

plt.tight_layout()
plt.show()

# Identificar el punto optimo
mejor_test_idx = test_scores.index(max(test_scores))  # indice del mejor test score
print(f"Mejor max_depth segun test F1: {profundidades[mejor_test_idx]}")
print(f"  Train F1: {train_scores[mejor_test_idx]:.4f}")
print(f"  Test  F1: {test_scores[mejor_test_idx]:.4f}")


## Que es un Pipeline de ML

```
PIPELINE DE ML
==============

Sin Pipeline (manual y propenso a errores):
  X_train -> StandardScaler.fit_transform() -> LogisticRegression.fit()
  X_test  -> StandardScaler.transform()     -> LogisticRegression.predict()
  RIESGO: si alguien llama .fit() en el scaler con X_test -> DATA LEAKAGE
          (el scaler aprende estadisticas del test, lo que invalida la evaluacion)

Con Pipeline (automatico y seguro):
  pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression())
  ])

  pipe.fit(X_train, y_train)
    -> INTERNAMENTE: scaler.fit_transform(X_train) + modelo.fit(X_scaled, y)

  pipe.predict(X_test)
    -> INTERNAMENTE: scaler.transform(X_test) + modelo.predict(X_scaled)
    -> El scaler NUNCA hace fit con datos de test

  Bonus: Pipeline funciona directamente con cross_val_score y GridSearchCV
  -> El scaler hace fit solo en cada fold de train, nunca en test fold
```


In [ ]:
# ============================================================
# BLOQUE: Construccion y entrenamiento de un Pipeline
# ============================================================

# Paso 1: construir el Pipeline como una lista de tuplas
# Cada tupla es (nombre_del_paso, objeto_transformador_o_modelo)
# Los nombres son importantes: se usan en GridSearchCV para referenciar params
pipeline = Pipeline([
    ("escalador", StandardScaler()),         # Paso 1: normalizar features a media=0, std=1
    ("modelo",    LogisticRegression(         # Paso 2: clasificador lineal
        max_iter=1000,   # iteraciones del solver para convergencia garantizada
        random_state=42  # reproducibilidad
    ))
])

# Paso 2: dividir X_cv en train y validation para demostrar el pipeline
X_tr, X_val, y_tr, y_val = train_test_split(
    X_cv, y_cv, test_size=0.2, random_state=42, stratify=y_cv
)

# Paso 3: entrenar el pipeline completo
# .fit() aplica TODOS los pasos en orden:
#   1. escalador.fit_transform(X_tr) -> X_scaled
#   2. modelo.fit(X_scaled, y_tr)
pipeline.fit(X_tr, y_tr)  # un solo .fit() lo hace todo, de forma segura
print("Pipeline entrenado correctamente")

# Paso 4: predecir sobre validacion
# .predict() aplica TODOS los pasos en orden:
#   1. escalador.transform(X_val) -> X_val_scaled (SIN .fit())
#   2. modelo.predict(X_val_scaled)
y_pred_pipe = pipeline.predict(X_val)  # predicciones sin riesgo de data leakage

# Paso 5: mostrar el reporte completo
print("\nClassification Report — Pipeline (StandardScaler + LogisticRegression):")
print("=" * 65)
print(
    classification_report(
        y_val, y_pred_pipe,
        target_names=["NO churn", "SI churn"]
    )
)


In [ ]:
# ============================================================
# BLOQUE: Pipeline dentro de cross_val_score
# ============================================================

# La combinacion de Pipeline + cross_val_score es la PRACTICA CORRECTA
# En cada fold:
#   - scaler.fit_transform() solo con los datos de TRAIN del fold
#   - scaler.transform() con los datos de TEST del fold
# Esto garantiza que NO hay data leakage en ninguno de los K folds

# Evaluar el pipeline con K-Fold cross validation
cv_scores_pipe = cross_val_score(
    pipeline,      # el pipeline completo (escalador + modelo)
    X_cv,          # features (sin escalar -- el pipeline escala internamente)
    y_cv,          # target
    cv=kfold,      # estrategia StratifiedKFold definida antes
    scoring="f1"   # metrica F1 de la clase positiva
)

# Mostrar los resultados de cada fold
print("Pipeline (StandardScaler + LogisticRegression) — CV Results:")
print("=" * 55)
for i, s in enumerate(cv_scores_pipe, 1):
    print(f"  Fold {i}: {s:.4f}")

print(f"\n  Media:  {cv_scores_pipe.mean():.4f}")
print(f"  Std:    {cv_scores_pipe.std():.4f}")

# Comparacion con el Decision Tree sin pipeline
print("\nComparacion:")
print(f"  Decision Tree (sin pipeline):   {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"  LogReg Pipeline:                {cv_scores_pipe.mean():.4f} +/- {cv_scores_pipe.std():.4f}")


## GridSearchCV: Busqueda automatica de hiperparametros

```
GRID SEARCH CROSS VALIDATION
=============================

Objetivo: encontrar los mejores hiperparametros automaticamente
          sin probarlos uno a uno a mano.

param_grid = {
  'max_depth':        [3, 5, 7],
  'min_samples_leaf': [2, 5, 10]
}

Combinaciones a probar: 3 valores x 3 valores = 9 combinaciones
Para cada combinacion: 5-fold CV = 45 entrenamientos en total

max_depth=3  min_samples_leaf=2   CV score = 0.78
max_depth=3  min_samples_leaf=5   CV score = 0.79
max_depth=3  min_samples_leaf=10  CV score = 0.77
max_depth=5  min_samples_leaf=2   CV score = 0.82
max_depth=5  min_samples_leaf=5   CV score = 0.83  <- MEJOR
max_depth=5  min_samples_leaf=10  CV score = 0.81
max_depth=7  min_samples_leaf=2   CV score = 0.80
max_depth=7  min_samples_leaf=5   CV score = 0.81
max_depth=7  min_samples_leaf=10  CV score = 0.80

Resultado: max_depth=5, min_samples_leaf=5 con CV score = 0.83
```


In [ ]:
# ============================================================
# BLOQUE: GridSearchCV sobre DecisionTreeClassifier
# ============================================================

# Paso 1: definir el espacio de busqueda de hiperparametros
# Cada clave es el nombre de un hiperparametro del modelo
# Cada valor es la lista de opciones a probar
param_grid = {
    "max_depth":         [3, 5, 7, 10],   # profundidades a comparar
    "min_samples_leaf":  [2, 5, 10],      # tamanos minimos de hoja
    "min_samples_split": [2, 5, 10],      # tamanos minimos para dividir nodo
}
# Total combinaciones: 4 x 3 x 3 = 36 combinaciones
# Con 5 folds: 36 x 5 = 180 entrenamientos en total

# Paso 2: crear el modelo base a optimizar
dt_para_grid = DecisionTreeClassifier(random_state=42)  # sin hiperparametros fijos

# Paso 3: instanciar GridSearchCV
# estimator: el modelo a optimizar
# param_grid: espacio de busqueda
# cv: estrategia de cross-validation (misma que antes)
# scoring: metrica a maximizar
# n_jobs=-1: usar todos los nucleos disponibles del procesador
# verbose=1: mostrar progreso durante la busqueda
grid_search = GridSearchCV(
    estimator=dt_para_grid,  # modelo base
    param_grid=param_grid,   # combinaciones a probar
    cv=kfold,                # estrategia de CV (StratifiedKFold 5 folds)
    scoring="f1",            # optimizar el F1 de la clase positiva
    n_jobs=-1,               # paralelizar en todos los nucleos
    verbose=1,               # mostrar progreso (cuantos fits se hicieron)
    refit=True               # re-entrenar el mejor modelo con TODOS los datos de CV
)

# Paso 4: ejecutar la busqueda (esto entrena 36*5 = 180 modelos)
print("Ejecutando GridSearchCV... (puede tardar unos segundos)")
grid_search.fit(X_cv, y_cv)  # probar todas las combinaciones con CV
print("\nBusqueda completada!")


In [ ]:
# ============================================================
# BLOQUE: Analizar resultados de GridSearchCV
# ============================================================

# .best_params_ contiene el diccionario con los hiperparametros ganadores
print("Mejores hiperparametros encontrados:")
print("=" * 45)
for param, valor in grid_search.best_params_.items():
    print(f"  {param}: {valor}")  # imprimir cada hiperparametro y su valor optimo

# .best_score_ es el F1 promedio del mejor modelo en CV (estimacion honesta)
print(f"\nMejor F1 promedio en CV: {grid_search.best_score_:.4f}")

# .best_estimator_ es el modelo ya listo para usar (refit con todos los datos de CV)
mejor_dt = grid_search.best_estimator_  # modelo ganador, ya entrenado
print(f"\nMejor estimador: {mejor_dt}")

# Convertir cv_results_ a DataFrame para explorar top combinaciones
df_resultados_grid = pd.DataFrame(grid_search.cv_results_)  # tabla completa de resultados

# Seleccionar las columnas mas relevantes
cols_ver = ["params", "mean_test_score", "std_test_score", "rank_test_score"]

# Mostrar el top 5 de combinaciones
top5 = df_resultados_grid.sort_values("rank_test_score").head(5)  # las 5 mejores
print("\nTop 5 combinaciones por F1 CV:")
print("=" * 60)
print(top5[cols_ver].to_string(index=False))


In [ ]:
# ============================================================
# BLOQUE: Evaluacion final del mejor modelo en el holdout set
# ============================================================

# ESTE ES EL MOMENTO DE LA VERDAD:
# Usamos por PRIMERA vez el holdout set (que nunca vio el modelo)
# Este score es la estimacion mas honesta del rendimiento real

# Paso 1: predecir sobre el holdout
# mejor_dt ya fue re-entrenado por GridSearchCV con todos los datos de CV
y_pred_holdout = mejor_dt.predict(X_holdout)  # predicciones sobre datos nunca vistos

# Paso 2: calcular el F1 en el holdout
f1_holdout = f1_score(y_holdout, y_pred_holdout, average="weighted")  # F1 final

# Paso 3: comparar CV score vs holdout score
# Si son similares -> el CV estimaba bien el rendimiento real
# Si hay gran diferencia -> puede haber alguna forma de data leakage o overfitting
print("Evaluacion final del mejor modelo:")
print("=" * 50)
print(f"  F1 en CV (durante desarrollo):  {grid_search.best_score_:.4f}")
print(f"  F1 en Holdout (evaluacion real): {f1_holdout:.4f}")
diferencia = abs(grid_search.best_score_ - f1_holdout)  # brecha entre estimacion y realidad
print(f"  Diferencia:                      {diferencia:.4f}")

if diferencia < 0.05:
    print("  -> La estimacion CV fue precisa. Modelo confiable.")
elif diferencia < 0.10:
    print("  -> Diferencia moderada. Revisar si hay optimismo en el CV.")
else:
    print("  -> Diferencia grande. Posible sobreoptimizacion en la busqueda.")

print("\nReporte completo en holdout set:")
print(classification_report(
    y_holdout, y_pred_holdout,
    target_names=["NO churn", "SI churn"]
))


## Resumen de la Clase 11

**Checklist — lo que aprendiste hoy:**

- [ ] Por que un solo split es insuficiente (variabilidad del score segun seed)
- [ ] `StratifiedKFold`: K divisiones que preservan el balance de clases
- [ ] `cross_val_score`: obtener K scores con un solo comando
- [ ] Visualizar scores por fold para detectar inconsistencias
- [ ] Diagnosticar overfitting: train >> test (reducir complejidad)
- [ ] Diagnosticar underfitting: ambos bajos (aumentar complejidad)
- [ ] `Pipeline`: encadenar preprocesamiento + modelo de forma segura
- [ ] Combinar `Pipeline` con `cross_val_score` para evaluacion correcta
- [ ] `GridSearchCV`: buscar los mejores hiperparametros automaticamente
- [ ] Evaluar el modelo final en el **holdout set** (estimacion honesta)

**Proxima clase:** Proyecto Final — analisis completo de ventas integrando todo lo aprendido.
